# Interpretable Latent Structure in Housing Prices: Factor Analysis and Prediction in the Ames Housing Data
Giandomenico Porfidia, Matthew Liew, Robert Kennedy

## 1. Introduction and Research Question

The Ames housing data contain dozens of overlapping measures of size, quality, age, garage attributes, basement characteristics, neighborhood, and sale conditions, many of which are highly correlated. This raises a central question: **can the main structure of housing value be summarized by a small number of interpretable latent dimensions, and how much predictive accuracy is lost when we replace the full feature set with those dimensions?** To answer this, we first examine the empirical structure of the housing variables, then use PCA and Factor Analysis to determine whether the data contain a low-dimensional representation of housing characteristics. Finally, we compare full-feature regularized regression models with a model using only the extracted factor scores. This allows us to evaluate the tradeoff between predictive accuracy and interpretability: whether a compact set of housing constructs can provide a plausible, communicable explanation of sale prices while retaining most of the predictive power of a much larger model.


## 2. Data and Empirical Motivation

The cleaned Ames dataset contains 2,927 observations and 79 features: 36 numerical variables, 23 categorical variables, and 20 ordinal variables. The raw sale-price distribution is strongly right-skewed, so the predictive models later use `log(SalePrice)` as the outcome. Before modeling, the key empirical fact is that many of the strongest price predictors are not independent signals. Measures of quality, size, garage capacity, basement area, and year built move together, suggesting that the dataset may contain a smaller number of latent housing characteristics.


<div style="text-align:center; margin: 0.6em 0 0.2em 0;">
  <img src="./figures/top_correlations.png" alt="Figure 1: Top numeric and ordinal correlations with SalePrice." style="max-width: 600px; width: 88%; height: auto;">
</div>

*Figure 1: Top numeric and ordinal correlations with SalePrice.*


Figure 1 motivates the dimensionality-reduction analysis. The strongest correlations with sale price are concentrated in quality, size, garage, and age-related variables. This suggests that individual predictors may be manifestations of broader constructs such as overall quality/newness, living space, and garage capacity. The remaining exploratory plots are moved to the appendix so that the main paper can focus on the evidence directly needed to answer the research question.


## 3. PCA as a Baseline

PCA provides a useful baseline because it asks whether the 56 numerical and ordinal variables can be compressed into a small number of orthogonal directions. If the dataset had one or two dominant axes of variation, PCA would reveal a sharp scree-plot elbow and a small number of components would explain most variance. Instead, the PCA results show that housing characteristics are related but not reducible to a single dominant dimension.


<div style="text-align:center; margin: 0.6em 0 0.2em 0;">
  <img src="./figures/fig1_pca_variance.png" alt="Figure 2: PCA variance decomposition." style="max-width: 610px; width: 88%; height: auto;">
</div>

*Figure 2: PCA variance decomposition.*


The first principal component explains only 18.7% of total variance, and 24 principal components are needed to explain 80% of the variance. This means PCA confirms that the variables are correlated, but it does not provide a compact or highly interpretable representation. The first few PCs mix several concepts at once — quality, garage, size, basement, and age — rather than separating them into clean substantive dimensions. This motivates Factor Analysis, which is better aligned with the goal of recovering interpretable latent constructs rather than merely maximizing total variance explained.


## 4. Factor Analysis: Recovering Interpretable Housing Constructs

Factor Analysis is the central method of the paper because it directly models shared covariance among variables while allowing feature-specific noise to remain outside the factor structure. We therefore use FA to ask whether the Ames variables reflect a smaller set of underlying housing attributes. The number of factors is chosen by combining information criteria, total communality, and interpretability rather than relying mechanically on a single criterion.


<div style="text-align:center; margin: 0.6em 0 0.2em 0;">
  <img src="./figures/fig3_fa_selection.png" alt="Figure 3: Factor Analysis selection criteria." style="max-width: 610px; width: 88%; height: auto;">
</div>

*Figure 3: Factor Analysis selection criteria.*


The selection criteria do not point to a single obvious value of `k`: AIC and BIC continue rewarding additional factors for a substantial range. This is why the final choice must also consider interpretability. A seven-factor Varimax solution provides the best main specification: it is compact, stable, and recovers recognizable housing constructs. A fifteen-factor solution improves total communality, but much of that gain comes from doublets, single-feature factors, and noise absorption rather than from new core dimensions. The k=15 robustness check is reported in the appendix.


<div style="text-align:center; margin: 0.6em 0 0.2em 0;">
  <img src="./figures/fig5_fa_primary_k7.png" alt="Figure 4: Primary k=7 Varimax Factor Analysis solution." style="max-width: 650px; width: 88%; height: auto;">
</div>

*Figure 4: Primary k=7 Varimax Factor Analysis solution.*


The k=7 Varimax solution is interpretable and domain-aligned. The major factors can be summarized as **Quality/Newness**, **Garage**, **Ground Floor/Footprint**, **Vertical Living Space**, **Basement Finish**, **Pool**, and **Basement Condition**. Five of these are broad, nameable housing constructs; Pool is a two-variable doublet, and Basement Condition is weaker but still interpretable. Together, these factors capture the shared structure of the numerical and ordinal housing characteristics. Just as importantly, the model identifies variables with low communalities — such as some porch, temporal, and miscellaneous features — as idiosyncratic rather than forcing them into artificial dimensions.

This is a substantively stronger result than PCA for the purpose of explanation. PCA required many components and produced mixed directions; FA produces a small set of constructs that correspond to how homes are naturally described and valued.


## 5. Predictive Validation

The final step is to test whether the factors are merely descriptive or also predictively useful. We first fit full-feature Ridge and Lasso models using the one-hot-encoded feature matrix. These models represent the high-accuracy benchmark because they can use all numerical, ordinal, and categorical information, including neighborhood effects. We then compare them to a Ridge model that uses only the seven FA scores.


<div style="text-align:center; margin: 0.6em 0 0.2em 0;">
  <img src="./figures/fig9_reg_predictions.png" alt="Figure 5: Full-feature Ridge and Lasso actual vs. predicted sale price." style="max-width: 640px; width: 88%; height: auto;">
</div>

*Figure 5: Full-feature Ridge and Lasso actual vs. predicted sale price.*


The full-feature Ridge and Lasso models perform similarly, with test-set $R^2$ near 0.88 on the log-price scale and MAPE around 9.2%. Ridge keeps all 224 features with shrinkage, while Lasso achieves nearly the same performance using 107 active features. This suggests that much of the predictive signal is concentrated in a smaller subset of variables, although the full feature matrix still improves accuracy relative to the factor-only model.


**Table 1: Predictive performance comparison.**

| Model | Features used | $R^2$ (log) | RMSE (log) | MAPE | RMSE (dollars) |
|---|---:|---:|---:|---:|---:|
| FA-only Ridge | 7 | 0.8139 | 0.1793 | 12.3% | \$78,582 |
| Ridge full | 224 | 0.8841 | 0.1415 | 9.2% | \$48,242 |
| Lasso full | 107 | 0.8832 | 0.1421 | 9.2% | \$51,772 |


The table shows the central tradeoff. The seven-factor model explains about 81% of the test-set variation in log sale price, compared with about 88% for the full Ridge and Lasso models. Thus, replacing 224 full-model features with seven interpretable factor scores costs about seven percentage points of $R^2$ and about three percentage points of MAPE. The factor-only model also has a substantially higher dollar RMSE, which is expected because it cannot use neighborhood indicators or idiosyncratic features that matter especially for expensive homes. Nevertheless, the FA model preserves most of the predictive signal while providing a much clearer explanation of the main housing dimensions that drive price.


## 6. Conclusion

This analysis shows that the Ames housing data contain a clear but not one-dimensional latent structure. PCA confirms that the numerical and ordinal features are correlated, but it does not yield a compact or easily interpretable representation: many components are needed to explain most of the variance. Factor Analysis, by contrast, recovers a small set of interpretable housing constructs, including quality/newness, garage quality, footprint, vertical living space, basement finish, pool presence, and basement condition. These factors are not only descriptively meaningful but also predictively useful: a Ridge model using only seven factor scores achieves an $R^2$ of 0.8139, compared with 0.8841 for the full 224-feature Ridge model. Thus, the full model remains preferable for maximum predictive accuracy, especially for high-priced homes and neighborhood-specific premiums, but the factor model provides a substantially more interpretable explanation of sale prices while preserving most of the predictive signal.


# Appendix

The appendix contains additional figures, robustness checks, and all executable code. These materials support the main paper but are not necessary for the six-page narrative.


## Appendix A: Additional Exploratory Data Analysis


<div style="text-align:center; margin: 0.6em 0 0.2em 0;">
  <img src="./figures/saleprice_hist.png" alt="Appendix Figure A1: Distribution of sale prices." style="max-width: 520px; width: 88%; height: auto;">
</div>

*Appendix Figure A1: Distribution of sale prices.*


<div style="text-align:center; margin: 0.6em 0 0.2em 0;">
  <img src="./figures/numeric_scatterplots.png" alt="Appendix Figure A2: SalePrice versus selected numeric variables." style="max-width: 700px; width: 88%; height: auto;">
</div>

*Appendix Figure A2: SalePrice versus selected numeric variables.*


<div style="text-align:center; margin: 0.6em 0 0.2em 0;">
  <img src="./figures/categorical_boxplots.png" alt="Appendix Figure A3: SalePrice by selected categorical variables." style="max-width: 700px; width: 88%; height: auto;">
</div>

*Appendix Figure A3: SalePrice by selected categorical variables.*


<div style="text-align:center; margin: 0.6em 0 0.2em 0;">
  <img src="./figures/correlation_heatmap.png" alt="Appendix Figure A4: Correlation matrix for numerical and ordinal features." style="max-width: 700px; width: 88%; height: auto;">
</div>

*Appendix Figure A4: Correlation matrix for numerical and ordinal features.*


## Appendix B: PCA Details


<div style="text-align:center; margin: 0.6em 0 0.2em 0;">
  <img src="./figures/fig2_pca_loadings.png" alt="Appendix Figure B1: PCA loadings for the first five principal components." style="max-width: 700px; width: 88%; height: auto;">
</div>

*Appendix Figure B1: PCA loadings for the first five principal components.*


## Appendix C: Factor Analysis Robustness


<div style="text-align:center; margin: 0.6em 0 0.2em 0;">
  <img src="./figures/fig4_fa_rotation_effect.png" alt="Appendix Figure C1: Effect of Varimax rotation at k=7." style="max-width: 700px; width: 88%; height: auto;">
</div>

*Appendix Figure C1: Effect of Varimax rotation at k=7.*


<div style="text-align:center; margin: 0.6em 0 0.2em 0;">
  <img src="./figures/fig6_fa_robustness_k15.png" alt="Appendix Figure C2: k=15 Varimax robustness check." style="max-width: 700px; width: 88%; height: auto;">
</div>

*Appendix Figure C2: k=15 Varimax robustness check.*


<div style="text-align:center; margin: 0.6em 0 0.2em 0;">
  <img src="./figures/fig7_fa_k7_vs_k15.png" alt="Appendix Figure C3: k=7 versus k=15 Factor Analysis comparison." style="max-width: 700px; width: 88%; height: auto;">
</div>

*Appendix Figure C3: k=7 versus k=15 Factor Analysis comparison.*


The k=15 solution is useful as a robustness check. It raises total communality from about 23.1 to about 31.6, but many of the additional factors are doublets, singletons, or noise sinks. The main housing constructs from k=7 remain stable, which supports using the more parsimonious seven-factor solution in the main paper.


## Appendix D: Regression Details


<div style="text-align:center; margin: 0.6em 0 0.2em 0;">
  <img src="./figures/fig8_lasso_coefs.png" alt="Appendix Figure D1: Largest non-zero Lasso coefficients." style="max-width: 700px; width: 88%; height: auto;">
</div>

*Appendix Figure D1: Largest non-zero Lasso coefficients.*


<div style="text-align:center; margin: 0.6em 0 0.2em 0;">
  <img src="./figures/fig10_fa_only_predictions.png" alt="Appendix Figure D2: FA-only actual versus predicted sale price." style="max-width: 620px; width: 88%; height: auto;">
</div>

*Appendix Figure D2: FA-only actual versus predicted sale price.*


The Lasso coefficient plot provides detail on which full-model predictors remain active after L1 regularization. The FA-only prediction plot is moved here because the compact comparison table in the main text communicates the key result more efficiently.


## Appendix E: Code

All executable cells from the original notebook are preserved below. Running this appendix from top to bottom regenerates the figures and model outputs used in the paper.


In [1]:
import sys
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.metrics import accuracy_score, classification_report, ConfusionMatrixDisplay

import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size'] = 11
sns.set_style('whitegrid')

os.makedirs('figures', exist_ok=True)


In [2]:
from cleaning_preprocessing import preproccesing

df = pd.read_csv('ames_housing.csv')
ames, column_breakdown = preproccesing(df)

In [3]:
# Note that I have noticed that BsmtFin_Type_2 was not included in the ordinal map from Mat's code I adjust it here

bsmt_fin_type_map = {'None': 0, 'Unf': 1, 'LwQ': 2, 'Rec': 3, 'BLQ': 4, 'ALQ': 5, 'GLQ': 6}
ames['BsmtFin_Type_2'] = ames['BsmtFin_Type_2'].fillna('None').map(bsmt_fin_type_map)


In [4]:
categorical_vars = [col for fam in column_breakdown.values() for col in fam.get('categorical', [])]
numerical_vars = [col for fam in column_breakdown.values() for col in fam.get('numerical', [])]
ordinal_vars = [col for fam in column_breakdown.values() for col in fam.get('ordinal', [])]
all_feature_vars = categorical_vars + numerical_vars + ordinal_vars

In [5]:
mask = ames[all_feature_vars].notna().all(axis=1)
ames = ames[mask].reset_index(drop=True)

In [6]:
y_cont = ames['SalePrice']  # continuous (regression target)
y_cat  = (ames['HighSalePrice'] == 'High').astype(int)  # 1 = above median

In [7]:
# print(f"Observations : {len(ames):,}")
# print(f"Features total : {len(all_feature_vars)}")
# print(f"  Categorical : {len(categorical_vars)}")
# print(f"  Numerical : {len(numerical_vars)}")
# print(f"  Ordinal : {len(ordinal_vars)}")
# print(f"High-price homes (above median SalePrice): {y_cat.mean():.1%}")

In [8]:
# EDA data frame: raw cleaned features before one-hot encoding
X_eda = ames[all_feature_vars].copy()

overview = pd.DataFrame({
    "dtype": X_eda.dtypes,
    "non_missing": X_eda.notna().sum(),
    "missing": X_eda.isna().sum(),
    "missing_rate": X_eda.isna().mean(),
    "n_unique": X_eda.nunique()
}).sort_values("missing_rate", ascending=False)

overview.head(15)

,dtype,non_missing,missing,missing_rate,n_unique
MS_Zoning,str,2927,0,0.0,7
BsmtFin_SF_1,float64,2927,0,0.0,995
Mo_Sold_Cos,float64,2927,0,0.0,11
Mo_Sold_Sin,float64,2927,0,0.0,11
Yr_Sold,int64,2927,0,0.0,5
Bsmt_Half_Bath,float64,2927,0,0.0,3
Bsmt_Full_Bath,float64,2927,0,0.0,4
Bsmt_Unf_SF,float64,2927,0,0.0,1137
BsmtFin_SF_2,float64,2927,0,0.0,274
Garage_Yr_Blt,float64,2927,0,0.0,103


In [9]:
summary = pd.DataFrame({
    "quantity": [
        "rows",
        "features",
        "numeric variables",
        "categorical variables",
        "ordinal variables"
    ],
    "count": [
        X_eda.shape[0],
        X_eda.shape[1],
        len(numerical_vars),
        len(categorical_vars),
        len(ordinal_vars)
    ]
})

summary

,quantity,count
0,rows,2927
1,features,79
2,numeric variables,36
3,categorical variables,23
4,ordinal variables,20


In [10]:
fig, ax = plt.subplots(figsize=(7, 5))

ax.hist(y_cont, bins=30, edgecolor="white")
ax.set_xlabel("SalePrice")
ax.set_ylabel("Count")
ax.set_title("Distribution of Sale Prices")

plt.tight_layout()
plt.savefig('figures/saleprice_hist.png', dpi=300, bbox_inches='tight')
plt.close()

In [11]:
selected_numeric_vars = [
    "Lot_Area",
    "Year_Built",
    "Overall_Qual",
    "Full_Bath",
    "Bedroom_AbvGr",
    "Garage_Cars"
]

df_plot = X_eda.copy()
df_plot["SalePrice"] = y_cont

fig, axes = plt.subplots(nrows=2, ncols=3, figsize=(15, 8))
axes = axes.flatten()

for i, var in enumerate(selected_numeric_vars):
    axes[i].scatter(df_plot[var], df_plot["SalePrice"], alpha=0.45, s=12)
    axes[i].set_xlabel(var)
    axes[i].set_ylabel("SalePrice")
    axes[i].set_title(f"SalePrice vs {var}")

plt.tight_layout()
plt.savefig('figures/numeric_scatterplots.png', dpi=300, bbox_inches='tight')
plt.close()

In [12]:
selected_categorical_vars = [
    "Neighborhood",
    "Bldg_Type",
    "House_Style",
    "Foundation",
    "Heating",
    "Central_Air"
]

fig, axes = plt.subplots(nrows=2, ncols=3, figsize=(16, 9))
axes = axes.flatten()

for i, var in enumerate(selected_categorical_vars):
    sns.boxplot(
        data=df_plot,
        x=var,
        y="SalePrice",
        ax=axes[i]
    )
    axes[i].set_xlabel(var)
    axes[i].set_ylabel("SalePrice")
    axes[i].set_title(f"SalePrice by {var}")
    axes[i].tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.savefig('figures/categorical_boxplots.png', dpi=300, bbox_inches='tight')
plt.close()

In [13]:
df_corr = X_eda.copy()
df_corr["SalePrice"] = y_cont

corrs = df_corr.corr(numeric_only=True)["SalePrice"].sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10, 5))

corrs.drop("SalePrice").head(15).plot(kind="bar", ax=ax)

ax.set_ylabel("Correlation with SalePrice")
ax.set_title("Top 15 Numeric / Ordinal Correlations with SalePrice")
ax.tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.savefig('figures/top_correlations.png', dpi=300, bbox_inches='tight')
plt.close()

In [14]:
corr_features = numerical_vars + ordinal_vars
corr_matrix = ames[corr_features].astype(float).corr()

fig, ax = plt.subplots(figsize=(14, 12))

sns.heatmap(
    corr_matrix,
    cmap="coolwarm",
    center=0,
    ax=ax,
    square=False,
    cbar_kws={"shrink": 0.75}
)

ax.set_title("Correlation Matrix for Numerical and Ordinal Features")

plt.tight_layout()
plt.savefig('figures/correlation_heatmap.png', dpi=300, bbox_inches='tight')
plt.close()

In [15]:
pca_features = numerical_vars + ordinal_vars  # 56 features

X_pca_input = ames[pca_features].astype(float)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_pca_input)

pca = PCA(random_state=42)
X_pca = pca.fit_transform(X_scaled)

evr = pca.explained_variance_ratio_
cumvar = np.cumsum(evr)

n_80 = int(np.argmax(cumvar >= 0.80)) + 1
n_90 = int(np.argmax(cumvar >= 0.90)) + 1

In [16]:
print(f"PCA input shape : {X_scaled.shape}")
print(f"PC1 explains : {evr[0]:.1%} of variance")
print(f"PC1+PC2 : {cumvar[1]:.1%}")
print(f"PC1–PC5 : {cumvar[4]:.1%}")
print(f"PCs for 80% var : {n_80}")
print(f"PCs for 90% var : {n_90}")

PCA input shape : (2927, 56)
PC1 explains : 18.7% of variance
PC1+PC2 : 25.8%
PC1–PC5 : 40.5%
PCs for 80% var : 24
PCs for 90% var : 32


In [17]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Scree
axes[0].bar(range(1, 17), evr[:16], alpha=0.75, color='steelblue')
axes[0].plot(range(1, 17), evr[:16], 'o-', color='navy', linewidth=1.8, markersize=5)
axes[0].set_xlabel('Principal Component')
axes[0].set_ylabel('Proportion of Variance Explained')
axes[0].set_title('(a) Scree Plot')
axes[0].set_xticks(range(1, 17))

# Cumulative variance
axes[1].plot(range(1, len(cumvar) + 1), cumvar, 'o-', color='navy',
             linewidth=1.8, markersize=4)
axes[1].axhline(0.80, color='darkorange', linestyle='--', linewidth=1.2,
                label=f'80% ({n_80} PCs)')
axes[1].axhline(0.90, color='crimson', linestyle='--', linewidth=1.2,
                label=f'90% ({n_90} PCs)')
axes[1].set_xlabel('Number of Principal Components')
axes[1].set_ylabel('Cumulative Variance Explained')
axes[1].set_title('(b) Cumulative Explained Variance')
axes[1].legend()
axes[1].set_xlim(0, len(cumvar) + 1)

plt.suptitle('Figure 1 — PCA Variance Decomposition (Ames Housing, 56 features)',
             fontsize=12, y=1.02)
plt.tight_layout()
plt.savefig('figures/fig1_pca_variance.png', dpi=300, bbox_inches='tight')
plt.close()

In [18]:
#Figure 2: Loadings heatmap for PC1–PC5
n_pcs_show = 5
loadings = pd.DataFrame(
    pca.components_[:n_pcs_show].T,
    index=pca_features,
    columns=[f'PC{i+1}' for i in range(n_pcs_show)]
)

plt.figure(figsize=(9, 14))
sns.heatmap(
    loadings,
    annot=True,
    fmt='.2f',
    cmap='RdBu_r',
    center=0,
    linewidths=0.3,
    cbar_kws={'shrink': 0.6, 'label': 'Loading'}
)
plt.title('Figure 2 — PCA Loadings: First 5 Principal Components\n'
          '(red = positive, blue = negative)', fontsize=12)
plt.xlabel('Principal Component')
plt.ylabel('Feature')
plt.tight_layout()
plt.savefig('figures/fig2_pca_loadings.png', dpi=300, bbox_inches='tight')
plt.close()

In [19]:
# Top contributors to each of the first 5 PCs
for pc in ['PC1', 'PC2', 'PC3', 'PC4', 'PC5']:
    top_pos = loadings[pc].nlargest(5)
    top_neg = loadings[pc].nsmallest(3)
    print(f'\n=== {pc} ({evr[int(pc[2:])-1]:.1%} variance) ===================')
    print('  Positive (high loading):')
    for feat, val in top_pos.items():
        print(f'    {feat:<25} {val:+.3f}')
    print('  Negative (high magnitude):')
    for feat, val in top_neg.items():
        print(f'    {feat:<25} {val:+.3f}')


=== PC1 (18.7% variance) ===================
  Positive (high loading):
    Overall_Qual              +0.257
    Garage_Cars               +0.243
    Garage_Area               +0.235
    Exter_Qual                +0.229
    Bsmt_Qual                 +0.221
  Negative (high magnitude):
    Lot_Shape                 -0.109
    Enclosed_Porch            -0.066
    Fence                     -0.059

=== PC2 (7.1% variance) ===================
  Positive (high loading):
    TotRms_AbvGrd             +0.340
    Bedroom_AbvGr             +0.319
    X2nd_Flr_SF               +0.316
    Gr_Liv_Area               +0.286
    Bsmt_Unf_SF               +0.232
  Negative (high magnitude):
    BsmtFin_Type_1            -0.264
    Bsmt_Full_Bath            -0.242
    BsmtFin_SF_1              -0.227

=== PC3 (5.6% variance) ===================
  Positive (high loading):
    Lot_Area                  +0.262
    X1st_Flr_SF               +0.250
    BsmtFin_SF_1              +0.244
    Bsmt_Full_Bath    

In [20]:
from sklearn.decomposition import FactorAnalysis

In [21]:
# Kaiser baseline 
corr_mat = np.corrcoef(X_scaled.T)
eig_corr = np.linalg.eigvalsh(corr_mat)[::-1]
n_kaiser = int((eig_corr > 1).sum())
print(f"Kaiser criterion (λ > 1): {n_kaiser} factors")

Kaiser criterion (λ > 1): 17 factors


In [22]:
n_samples, n_features = X_scaled.shape
k_ceiling = int((2*n_features + 1 - np.sqrt(8*n_features + 1)) / 2)  # = 44
k_max = min(k_ceiling - 2, 50)
n_range = range(1, k_max + 1)

print(f"Features: {n_features}, theoretical max factors: {k_ceiling}, scanning to k={k_max}")

Features: 56, theoretical max factors: 45, scanning to k=43


In [23]:
total_communality = []
log_likelihoods = []
aic_scores = []
bic_scores = []

for k in n_range:
    fa_k = FactorAnalysis(n_components=k, random_state=42, max_iter=1000)
    fa_k.fit(X_scaled)

    ll = fa_k.score(X_scaled) * n_samples
    log_likelihoods.append(ll)
    total_communality.append((1 - fa_k.noise_variance_).sum())
    n_params = n_features * k + n_features - k * (k - 1) / 2

    aic_scores.append(2 * n_params - 2 * ll)
    bic_scores.append(n_params * np.log(n_samples) - 2 * ll)

In [24]:
n_factors_aic = list(n_range)[int(np.argmin(aic_scores))]
n_factors_bic = list(n_range)[int(np.argmin(bic_scores))]

print(f"AIC-optimal: {n_factors_aic} factors")
print(f"BIC-optimal: {n_factors_bic} factors")

AIC-optimal: 29 factors
BIC-optimal: 20 factors


In [25]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

aic_arr = np.array(aic_scores)
bic_arr = np.array(bic_scores)
ks = np.array(list(n_range))

# (a) AIC curve
axes[0].plot(ks, aic_arr, 'o-', color='navy', linewidth=1.8, markersize=5,
             label='AIC')
axes[0].axvline(n_factors_aic, color='crimson', linestyle='--', linewidth=1.5,
                label=f'AIC-optimal: k = {n_factors_aic}')
axes[0].axvline(n_kaiser, color='darkorange', linestyle=':', linewidth=1.5,
                label=f'Kaiser: k = {n_kaiser}')


axes[0].scatter([n_factors_aic], [aic_arr.min()], s=120, facecolors='none',
                edgecolors='crimson', linewidths=2, zorder=5)
axes[0].set_xlabel('Number of Factors (k)')
axes[0].set_ylabel('AIC  (lower is better)')
axes[0].set_title('AIC')
axes[0].set_xticks(ks)
axes[0].legend(fontsize=9)
axes[0].grid(alpha=0.3)

# (b) BIC curve
axes[1].plot(ks, bic_arr, 'o-', color='darkgreen', linewidth=1.8, markersize=5,
             label='BIC')
axes[1].axvline(n_factors_bic, color='crimson', linestyle='--', linewidth=1.5,
                label=f'BIC-optimal: k = {n_factors_bic}  ← selected')
axes[1].axvline(n_kaiser, color='darkorange', linestyle=':', linewidth=1.5,
                label=f'Kaiser: k = {n_kaiser}')
axes[1].scatter([n_factors_bic], [bic_arr.min()], s=120, facecolors='none',
                edgecolors='crimson', linewidths=2, zorder=5)


near_best = ks[bic_arr - bic_arr.min() < 2]
if len(near_best) > 1:
    axes[1].axvspan(near_best.min() - 0.4, near_best.max() + 0.4,
                    color='gray', alpha=0.12,
                    label=f'ΔBIC < 2 (k ∈ {near_best.min()}–{near_best.max()})')
axes[1].set_xlabel('Number of Factors (k)')
axes[1].set_ylabel('BIC  (lower is better)')
axes[1].set_title('(b) Bayesian Information Criterion')
axes[1].set_xticks(ks)
axes[1].legend(fontsize=9)
axes[1].grid(alpha=0.3)

plt.suptitle(
    f'Figure 3 — Selecting Number of FA Factors via Information Criteria\n'
    f'Selected k = {n_factors_bic} (BIC); AIC suggests {n_factors_aic}; Kaiser suggests {n_kaiser}',
    fontsize=12, y=1.03
)
plt.tight_layout()
plt.savefig('figures/fig3_fa_selection.png', dpi=300, bbox_inches='tight')
plt.close()

# Summary
print(f"AIC-optimal k = {n_factors_aic}   (AIC = {aic_arr.min():.1f})")
print(f"BIC-optimal k = {n_factors_bic}   (BIC = {bic_arr.min():.1f})")
if len(near_best) > 1:
    print(f"Models within ΔBIC < 2 of best: k ∈ {list(near_best)}  "
          f"(statistically comparable)")

AIC-optimal k = 29   (AIC = 216542.0)
BIC-optimal k = 20   (BIC = 222865.9)


In [26]:
# Unrotated k=7
fa7 = FactorAnalysis(n_components=7, random_state=42, max_iter=1000)
fa7.fit(X_scaled)
load7_unrot = pd.DataFrame(
    fa7.components_.T,
    index=pca_features,
    columns=[f'F{i+1}' for i in range(7)]
)

In [27]:
# Varimax-rotated k=7 (primary solution)
fa7_rot = FactorAnalysis(n_components=7, rotation='varimax',
                          random_state=42, max_iter=1000)
fa7_rot.fit(X_scaled)

load7_rot = pd.DataFrame(
    fa7_rot.components_.T,
    index=pca_features,
    columns=[f'F{i+1}' for i in range(7)]
)
comm7_rot = pd.Series(1 - fa7_rot.noise_variance_, index=pca_features)

print(f"Total communality (k=7, Varimax) : {comm7_rot.sum():.2f} / {len(pca_features)}")
print(f"Mean communality                  : {comm7_rot.mean():.3f}")
print(f"h² > 0.5                          : {(comm7_rot > 0.5).sum()} features")
print(f"h² < 0.2                          : {(comm7_rot < 0.2).sum()} features")

Total communality (k=7, Varimax) : 23.11 / 56
Mean communality                  : 0.413
h² > 0.5                          : 23 features
h² < 0.2                          : 22 features


In [28]:
# Varimax-rotated k=15 
fa15_rot = FactorAnalysis(n_components=15, rotation='varimax',
                           random_state=42, max_iter=1000)
fa15_rot.fit(X_scaled)

load15_rot = pd.DataFrame(
    fa15_rot.components_.T,
    index=pca_features,
    columns=[f'F{i+1}' for i in range(15)]
)
comm15_rot = pd.Series(1 - fa15_rot.noise_variance_, index=pca_features)

print(f"Total communality (k=15, Varimax) : {comm15_rot.sum():.2f} / {len(pca_features)}")

Total communality (k=15, Varimax) : 31.56 / 56


In [29]:
def sort_by_primary(loadings):
    primary = loadings.abs().idxmax(axis=1)
    strength = loadings.abs().max(axis=1)
    order = (pd.DataFrame({'factor': primary, 'loading': strength})
             .sort_values(['factor', 'loading'], ascending=[True, False])
             .index)
    return order, primary

order_7,  primary_7  = sort_by_primary(load7_rot)
order_15, primary_15 = sort_by_primary(load15_rot)

In [30]:
# ═══════════════════════════════════════════════════════════════════════════
# FIGURE 4 — Rotation Effect (Unrotated vs Varimax at k=7)
# ═══════════════════════════════════════════════════════════════════════════

fig, axes = plt.subplots(1, 2, figsize=(16, 12))

# (a) Unrotated loadings
masked_unrot = load7_unrot.where(load7_unrot.abs() >= 0.3, 0)
sns.heatmap(
    masked_unrot, annot=True, fmt='.2f', annot_kws={'size': 7},
    cmap='RdBu_r', center=0, vmin=-1, vmax=1, linewidths=0.3,
    ax=axes[0], cbar_kws={'shrink': 0.6, 'label': 'Loading'}
)
axes[0].set_title('(a) Unrotated Loadings — k=7\n'
                  'F1 acts as a "g-factor" loading on nearly everything',
                  fontsize=10)
axes[0].set_xlabel('Factor')
axes[0].set_ylabel('Feature')
axes[0].tick_params(axis='y', labelsize=7)

# (b) Varimax rotated loadings (sorted)
load7_sorted = load7_rot.loc[order_7]
masked_rot = load7_sorted.where(load7_sorted.abs() >= 0.3, 0)
sns.heatmap(
    masked_rot, annot=True, fmt='.2f', annot_kws={'size': 7},
    cmap='RdBu_r', center=0, vmin=-1, vmax=1, linewidths=0.3,
    ax=axes[1], cbar_kws={'shrink': 0.6, 'label': 'Loading'}
)
axes[1].set_title('(b) Varimax-Rotated Loadings — k=7\n'
                  'Features sorted by primary factor → block-diagonal structure',
                  fontsize=10)
axes[1].set_xlabel('Factor')
axes[1].set_ylabel('')
axes[1].tick_params(axis='y', labelsize=7)

# Factor boundary lines on rotated heatmap
boundaries_7 = (primary_7.loc[order_7]
                .ne(primary_7.loc[order_7].shift())
                .cumsum().value_counts().sort_index().cumsum())
for b in boundaries_7[:-1]:
    axes[1].axhline(b, color='black', linewidth=1.5)

plt.suptitle('Figure 4 — Effect of Varimax Rotation at k=7  (|loading| ≥ 0.3 shown)',
             fontsize=12, y=1.00)
plt.tight_layout()
plt.savefig('figures/fig4_fa_rotation_effect.png', dpi=300, bbox_inches='tight')
plt.close()

In [31]:
# ═══════════════════════════════════════════════════════════════════════════
# FIGURE 5 — Primary Solution (k=7 Varimax): Communalities + Loadings + Variance
# ═══════════════════════════════════════════════════════════════════════════

comm7_sorted = comm7_rot.loc[order_7]

fig = plt.figure(figsize=(18, 13))
gs = fig.add_gridspec(1, 3, width_ratios=[1, 2, 1], wspace=0.4)
ax_comm = fig.add_subplot(gs[0, 0])
ax_load = fig.add_subplot(gs[0, 1])
ax_var  = fig.add_subplot(gs[0, 2])

# (a) Communalities
bar_colors = ['#e6550d' if v >= 0.5 else '#6baed6' if v >= 0.3 else '#d9d9d9'
              for v in comm7_sorted.values]
comm7_sorted.plot(kind='barh', color=bar_colors, ax=ax_comm)
ax_comm.axvline(0.5, color='crimson', linestyle='--', linewidth=1.2, label='h² = 0.5')
ax_comm.axvline(0.3, color='darkorange', linestyle='--', linewidth=1.2, label='h² = 0.3')
ax_comm.set_xlabel('Communality (h²)')
ax_comm.set_title(f'(a) Communalities (Varimax, k=7)\n'
                  f'Σh² = {comm7_rot.sum():.1f}/{len(pca_features)}, '
                  f'mean = {comm7_rot.mean():.2f}', fontsize=10)
ax_comm.legend(fontsize=8, loc='lower right')
ax_comm.tick_params(axis='y', labelsize=8)
ax_comm.invert_yaxis()

# (b) Loadings heatmap (sorted)
masked = load7_sorted.where(load7_sorted.abs() >= 0.3, 0)
sns.heatmap(
    masked, annot=True, fmt='.2f', annot_kws={'size': 8},
    cmap='RdBu_r', center=0, vmin=-1, vmax=1, linewidths=0.3,
    ax=ax_load, cbar_kws={'shrink': 0.6, 'label': 'Loading'}
)
ax_load.set_title('(b) Varimax-Rotated Loadings (|loading| ≥ 0.3 shown)\n'
                  'Features sorted by primary factor', fontsize=10)
ax_load.set_xlabel('Factor')
ax_load.set_ylabel('')
ax_load.tick_params(axis='y', labelsize=8)
for b in boundaries_7[:-1]:
    ax_load.axhline(b, color='black', linewidth=1.5)

# (c) Variance explained per factor
var_per_factor_7 = (load7_rot ** 2).sum(axis=0).sort_values(ascending=True)
var_pct_7 = var_per_factor_7 / len(pca_features) * 100
var_per_factor_7.plot(kind='barh', color='steelblue', ax=ax_var, alpha=0.85)
for i, (factor, val) in enumerate(var_per_factor_7.items()):
    ax_var.text(val + 0.1, i, f'{var_pct_7[factor]:.1f}%', va='center', fontsize=9)
ax_var.set_xlabel('Sum of squared loadings')
ax_var.set_title('(c) Variance Explained\nper Rotated Factor', fontsize=10)
ax_var.tick_params(axis='y', labelsize=9)

plt.suptitle('Figure 5 — Primary Solution: Varimax-Rotated FA at k=7',
             fontsize=12, y=1.00)
plt.tight_layout()
plt.savefig('figures/fig5_fa_primary_k7.png', dpi=300, bbox_inches='tight')
plt.close()

In [32]:
# ═══════════════════════════════════════════════════════════════════════════
# FIGURE 6 — Robustness Check (k=15 Varimax): Same three-panel layout
# ═══════════════════════════════════════════════════════════════════════════

comm15_sorted = comm15_rot.loc[order_15]
load15_sorted = load15_rot.loc[order_15]

fig = plt.figure(figsize=(22, 13))
gs = fig.add_gridspec(1, 3, width_ratios=[1, 3, 1], wspace=0.4)
ax_comm = fig.add_subplot(gs[0, 0])
ax_load = fig.add_subplot(gs[0, 1])
ax_var  = fig.add_subplot(gs[0, 2])

# (a) Communalities
bar_colors = ['#e6550d' if v >= 0.5 else '#6baed6' if v >= 0.3 else '#d9d9d9'
              for v in comm15_sorted.values]
comm15_sorted.plot(kind='barh', color=bar_colors, ax=ax_comm)
ax_comm.axvline(0.5, color='crimson', linestyle='--', linewidth=1.2, label='h² = 0.5')
ax_comm.axvline(0.3, color='darkorange', linestyle='--', linewidth=1.2, label='h² = 0.3')
ax_comm.set_xlabel('Communality (h²)')
ax_comm.set_title(f'(a) Communalities (Varimax, k=15)\n'
                  f'Σh² = {comm15_rot.sum():.1f}/{len(pca_features)}, '
                  f'mean = {comm15_rot.mean():.2f}', fontsize=10)
ax_comm.legend(fontsize=8, loc='lower right')
ax_comm.tick_params(axis='y', labelsize=7)
ax_comm.invert_yaxis()

# (b) Loadings heatmap
masked = load15_sorted.where(load15_sorted.abs() >= 0.3, 0)
sns.heatmap(
    masked, annot=True, fmt='.2f', annot_kws={'size': 7},
    cmap='RdBu_r', center=0, vmin=-1, vmax=1, linewidths=0.3,
    ax=ax_load, cbar_kws={'shrink': 0.6, 'label': 'Loading'}
)
ax_load.set_title('(b) Varimax-Rotated Loadings — k=15 (|loading| ≥ 0.3 shown)\n'
                  'Features sorted by primary factor', fontsize=10)
ax_load.set_xlabel('Factor')
ax_load.set_ylabel('')
ax_load.tick_params(axis='y', labelsize=7)

boundaries_15 = (primary_15.loc[order_15]
                 .ne(primary_15.loc[order_15].shift())
                 .cumsum().value_counts().sort_index().cumsum())
for b in boundaries_15[:-1]:
    ax_load.axhline(b, color='black', linewidth=1.5)

# (c) Variance per factor
var_per_factor_15 = (load15_rot ** 2).sum(axis=0).sort_values(ascending=True)
var_pct_15 = var_per_factor_15 / len(pca_features) * 100
var_per_factor_15.plot(kind='barh', color='steelblue', ax=ax_var, alpha=0.85)
for i, (factor, val) in enumerate(var_per_factor_15.items()):
    ax_var.text(val + 0.1, i, f'{var_pct_15[factor]:.1f}%', va='center', fontsize=9)
ax_var.set_xlabel('Sum of squared loadings')
ax_var.set_title('(c) Variance Explained\nper Rotated Factor', fontsize=10)
ax_var.tick_params(axis='y', labelsize=9)

plt.suptitle('Figure 6 — Robustness Check: Varimax-Rotated FA at k=15',
             fontsize=12, y=1.00)
plt.tight_layout()
plt.savefig('figures/fig6_fa_robustness_k15.png', dpi=300, bbox_inches='tight')
plt.close()

In [33]:
# ═══════════════════════════════════════════════════════════════════════════
# FIGURE 7 — k=7 vs k=15 Comparison: communality gain and where it goes
# ═══════════════════════════════════════════════════════════════════════════

delta_comm = (comm15_rot - comm7_rot).sort_values(ascending=True)

fig, axes = plt.subplots(1, 2, figsize=(15, 10))

# (a) Side-by-side communality bars
comp_df = pd.DataFrame({'k=7': comm7_rot, 'k=15': comm15_rot}).sort_values('k=15')
comp_df.plot(kind='barh', ax=axes[0], color=['#6baed6', '#e6550d'],
             alpha=0.85, width=0.8)
axes[0].axvline(0.5, color='crimson', linestyle='--', linewidth=1.0, alpha=0.6)
axes[0].axvline(0.3, color='darkorange', linestyle='--', linewidth=1.0, alpha=0.6)
axes[0].set_xlabel('Communality (h²)')
axes[0].set_title(f'(a) Communality: k=7 vs k=15\n'
                  f'Σh² gain: {comm15_rot.sum() - comm7_rot.sum():.1f} '
                  f'(+{(comm15_rot.sum() - comm7_rot.sum())/len(pca_features)*100:.1f} pp)',
                  fontsize=10)
axes[0].legend(loc='lower right', fontsize=9)
axes[0].tick_params(axis='y', labelsize=7)

# (b) Δ communality — which features benefit from extra factors
colors = ['#e6550d' if v >= 0.15 else '#6baed6' if v >= 0.05 else '#d9d9d9'
          for v in delta_comm.values]
delta_comm.plot(kind='barh', color=colors, ax=axes[1])
axes[1].axvline(0, color='black', linewidth=0.8)
axes[1].set_xlabel('Δ h²  (k=15 minus k=7)')
axes[1].set_title('(b) Communality Gain per Feature\n'
                  'Orange: large gain (likely captured by new useful factors)\n'
                  'Gray: little/no gain (idiosyncratic features)',
                  fontsize=10)
axes[1].tick_params(axis='y', labelsize=7)

plt.suptitle('Figure 7 — What Do the Extra 8 Factors at k=15 Buy Us?',
             fontsize=12, y=1.01)
plt.tight_layout()
plt.savefig('figures/fig7_fa_k7_vs_k15.png', dpi=300, bbox_inches='tight')
plt.close()

In [34]:
# Extract factor scores for downstream modeling
fa_scores = pd.DataFrame(
    fa7_rot.transform(X_scaled),
    columns=['Vertical_Size', 'Footprint', 'Bsmt_Finish',
             'Garage', 'Bsmt_Cond', 'Pool', 'Quality_Newness'],
    index=ames.index
)
fa_scores['Quality_Newness'] *= -1
fa_scores['Garage'] *= -1

print(f"Factor scores extracted: {fa_scores.shape}")
fa_scores.head()

Factor scores extracted: (2927, 7)


,Vertical_Size,Footprint,Bsmt_Finish,Garage,Bsmt_Cond,Pool,Quality_Newness
0,-0.182069,1.310735,0.334709,0.325237,1.079606,-0.120515,-0.761174
1,-0.933461,-0.434637,0.407347,0.386872,-0.351501,0.004934,-0.848261
2,-0.470763,0.803307,0.611647,0.269518,-0.700045,-0.128807,-1.013262
3,0.070622,2.602091,-0.171882,0.134990,-0.386987,-0.217876,0.027505
4,0.648941,-0.681809,0.994871,0.270343,-0.397366,-0.072286,0.121456


In [35]:
print("\n" + "="*72)
print("FACTOR INTERPRETATION — k=7 Varimax (PRIMARY SOLUTION)")
print("="*72)
factor_names_map = {
    'F1': 'Vertical_Size', 'F2': 'Footprint', 'F3': 'Bsmt_Finish',
    'F4': 'Garage', 'F5': 'Bsmt_Cond', 'F6': 'Pool', 'F7': 'Quality_Newness'
}
for factor in load7_rot.columns:
    factor_loads = load7_rot[factor]
    strong = factor_loads[factor_loads.abs() >= 0.4].sort_values(key=abs, ascending=False)
    pct = var_pct_7.get(factor, 0)
    name = factor_names_map.get(factor, '?')
    print(f"\n{factor} — {name}  ({pct:.1f}% variance):")
    if len(strong) == 0:
        print("  (no features with |loading| ≥ 0.4 — weak factor)")
    else:
        for feat, val in strong.items():
            print(f"  {feat:25s}  {val:+.2f}")

print("\n" + "="*72)
print(f"FEATURES WITH h² < 0.3  (poorly explained — candidates to drop or keep raw)")
print("="*72)
poor = comm7_rot[comm7_rot < 0.3].sort_values()
for feat, val in poor.items():
    print(f"  {feat:25s}  h² = {val:.2f}")

print(f"\nTotal: {len(poor)} of {len(pca_features)} features below h² = 0.3")


FACTOR INTERPRETATION — k=7 Varimax (PRIMARY SOLUTION)

F1 — Vertical_Size  (6.6% variance):
  X2nd_Flr_SF                +0.93
  Gr_Liv_Area                +0.84
  TotRms_AbvGrd              +0.74
  Bedroom_AbvGr              +0.60
  Half_Bath                  +0.55
  Full_Bath                  +0.47

F2 — Footprint  (6.7% variance):
  X1st_Flr_SF                +0.95
  Total_Bsmt_SF              +0.81
  Gr_Liv_Area                +0.45
  BsmtFin_SF_1               +0.44

F3 — Bsmt_Finish  (4.7% variance):
  Bsmt_Unf_SF                -0.89
  BsmtFin_SF_1               +0.78
  BsmtFin_Type_1             +0.62
  Bsmt_Full_Bath             +0.60

F4 — Garage  (7.1% variance):
  Garage_Yr_Blt              -0.97
  Garage_Cond                -0.97
  Garage_Qual                -0.96
  Garage_Cars                -0.52
  Garage_Area                -0.50
  Garage_Finish              -0.43

F5 — Bsmt_Cond  (1.9% variance):
  Bsmt_Cond                  -0.59
  Bsmt_Qual                  -0.43
 

In [36]:
# One-hot encode categorical variables
X_rf = ames[all_feature_vars].copy()
X_rf = pd.get_dummies(X_rf, columns=categorical_vars, drop_first=True)
X_rf = X_rf.astype(float)

y_log = np.log(y_cont)

X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(
    X_rf, y_log, test_size=0.20, random_state=42
)

In [37]:
print(f"Regression target : log(SalePrice)  — corrects right skew")
print(f"Train : {X_train_r.shape[0]:,}  |  Test : {X_test_r.shape[0]:,}  |  Features : {X_rf.shape[1]}")
print(f"\nlog(SalePrice) — mean: {y_log.mean():.3f}  std: {y_log.std():.3f}")
print(f"SalePrice       — median: ${np.exp(np.median(y_log)):,.0f}  mean: ${np.exp(y_log.mean()):,.0f}")

Regression target : log(SalePrice)  — corrects right skew
Train : 2,341  |  Test : 586  |  Features : 224

log(SalePrice) — mean: 12.021  std: 0.407
SalePrice       — median: $160,000  mean: $166,280


In [38]:
from sklearn.linear_model import LassoCV, RidgeCV
from sklearn.metrics import r2_score, mean_squared_error

In [39]:
# Scale features (required for regularised regression — coefficients become comparable)
reg_scaler = StandardScaler()
X_train_s  = reg_scaler.fit_transform(X_train_r)
X_test_s   = reg_scaler.transform(X_test_r)

In [40]:
# ridge

ridge_cv = RidgeCV(alphas=np.logspace(-2, 4, 100), cv=5)
ridge_cv.fit(X_train_s, y_train_r)
y_pred_ridge = ridge_cv.predict(X_test_s)

In [41]:
# lasso

lasso_cv = LassoCV(cv=5, random_state=42, max_iter=10000, n_alphas=100)
lasso_cv.fit(X_train_s, y_train_r)
y_pred_lasso = lasso_cv.predict(X_test_s)

In [42]:
# metrics

r2_ridge   = r2_score(y_test_r, y_pred_ridge)
r2_lasso   = r2_score(y_test_r, y_pred_lasso)
rmse_ridge = np.sqrt(mean_squared_error(y_test_r, y_pred_ridge))
rmse_lasso = np.sqrt(mean_squared_error(y_test_r, y_pred_lasso))

In [43]:
# original scale

y_pr_usd = np.exp(y_pred_ridge);  y_pl_usd = np.exp(y_pred_lasso);  y_t_usd = np.exp(y_test_r)
mape_ridge     = np.mean(np.abs((y_t_usd - y_pr_usd) / y_t_usd)) * 100
mape_lasso     = np.mean(np.abs((y_t_usd - y_pl_usd) / y_t_usd)) * 100
rmse_ridge_usd = np.sqrt(mean_squared_error(y_t_usd, y_pr_usd))
rmse_lasso_usd = np.sqrt(mean_squared_error(y_t_usd, y_pl_usd))

n_nonzero_lasso = (np.abs(lasso_cv.coef_) >= 1e-8).sum()
n_zero_lasso    = (np.abs(lasso_cv.coef_) <  1e-8).sum()

In [44]:
print(f"{'Model':<8} {'α':>10} {'R² (log)':>10} {'RMSE (log)':>12} {'MAPE':>8} {'RMSE ($)':>12} {'Active':>8}")
print("-" * 75)
print(f"{'Ridge':<8} {ridge_cv.alpha_:>10.2f} {r2_ridge:>10.4f} {rmse_ridge:>12.4f} "
      f"{mape_ridge:>7.1f}% ${rmse_ridge_usd:>10,.0f} {'all':>8}")
print(f"{'Lasso':<8} {lasso_cv.alpha_:>10.5f} {r2_lasso:>10.4f} {rmse_lasso:>12.4f} "
      f"{mape_lasso:>7.1f}% ${rmse_lasso_usd:>10,.0f} {n_nonzero_lasso:>8}")
print(f"\nLasso zeroed out {n_zero_lasso}/{X_rf.shape[1]} features ({n_zero_lasso/X_rf.shape[1]:.0%})")

Model             α   R² (log)   RMSE (log)     MAPE     RMSE ($)   Active
---------------------------------------------------------------------------
Ridge        265.61     0.8841       0.1415     9.2% $    48,242      all
Lasso       0.00271     0.8832       0.1421     9.2% $    51,772      107

Lasso zeroed out 117/224 features (52%)


In [45]:
# Top Lasso coefficients (non-zero only)
lasso_coefs = pd.Series(lasso_cv.coef_, index=X_rf.columns)
nonzero     = lasso_coefs[lasso_coefs.abs() > 1e-8]

top_pos  = nonzero.nlargest(12)
top_neg  = nonzero.nsmallest(8)
top_show = pd.concat([top_neg, top_pos]).sort_values()

colors_c = ['#3182bd' if v < 0 else '#e6550d' for v in top_show.values]

fig, ax = plt.subplots(figsize=(10, 7))
top_show.plot(kind='barh', color=colors_c, ax=ax, edgecolor='white', linewidth=0.4)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_xlabel('Standardised Lasso Coefficient  (log SalePrice scale)')
ax.set_title(f'Figure 8 — Lasso Regression: Largest Coefficients\n'
             f'α = {lasso_cv.alpha_:.5f}  |  {n_nonzero_lasso} active features of {X_rf.shape[1]}',
             fontsize=12)
patches_c = [
    mpatches.Patch(color='#e6550d', label='Positive — associated with higher price'),
    mpatches.Patch(color='#3182bd', label='Negative — associated with lower price')
]
ax.legend(handles=patches_c, fontsize=9)
plt.tight_layout()
plt.savefig('figures/fig8_lasso_coefs.png', dpi=300, bbox_inches='tight')
plt.close()

In [46]:
y_pred_ridge_usd = np.exp(y_pred_ridge)
y_pred_lasso_usd = np.exp(y_pred_lasso)
y_test_usd       = np.exp(y_test_r)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for i, (y_pred_usd, model_name, r2_val) in enumerate([
    (y_pred_ridge_usd, 'Ridge', r2_ridge),
    (y_pred_lasso_usd, 'Lasso', r2_lasso)
]):
    axes[i].scatter(y_test_usd / 1000, y_pred_usd / 1000,
                    alpha=0.4, s=15, color='steelblue', rasterized=True)
    lo = min(y_test_usd.min(), y_pred_usd.min()) / 1000 * 0.93
    hi = max(y_test_usd.max(), y_pred_usd.max()) / 1000 * 1.05
    axes[i].plot([lo, hi], [lo, hi], 'r--', linewidth=1.5, label='Perfect prediction')
    axes[i].set_xlabel('Actual Sale Price ($k)')
    axes[i].set_ylabel('Predicted Sale Price ($k)')
    axes[i].set_title(f'({chr(97+i)}) {model_name}   R² = {r2_val:.3f}')
    axes[i].legend(fontsize=9)

plt.suptitle('Figure 9 — Actual vs Predicted Sale Price (test set, original scale)',
             fontsize=12, y=1.02)
plt.tight_layout()
plt.savefig('figures/fig9_reg_predictions.png', dpi=300, bbox_inches='tight')
plt.close()

In [47]:
X_fa_input_full = ames[pca_features].astype(float)

X_fa_train_raw, X_fa_test_raw, y_train_fa, y_test_fa = train_test_split(
    X_fa_input_full, y_log, test_size=0.20, random_state=42
)

# Scale using train-only stats
fa_scaler = StandardScaler()
X_fa_train = fa_scaler.fit_transform(X_fa_train_raw)
X_fa_test  = fa_scaler.transform(X_fa_test_raw)

# Fit FA on training data only
fa_holdout = FactorAnalysis(n_components=7, rotation='varimax',
                             random_state=42, max_iter=1000)
fa_holdout.fit(X_fa_train)

# Transform both sets
fa_scores_train = fa_holdout.transform(X_fa_train)
fa_scores_test  = fa_holdout.transform(X_fa_test)

print(f"FA refit on training only: {X_fa_train.shape[0]:,} samples, {X_fa_train.shape[1]} features")
print(f"Factor scores: train {fa_scores_train.shape}, test {fa_scores_test.shape}")
print(f"Training communality: {(1 - fa_holdout.noise_variance_).sum():.2f} / {len(pca_features)}")

FA refit on training only: 2,341 samples, 56 features
Factor scores: train (2341, 7), test (586, 7)
Training communality: 22.93 / 56


In [48]:
# Fit Ridge on the 7 factor scores
ridge_fa = RidgeCV(alphas=np.logspace(-3, 3, 100), cv=5)
ridge_fa.fit(fa_scores_train, y_train_fa)

y_pred_fa = ridge_fa.predict(fa_scores_test)

# Metrics on log scale
r2_fa   = r2_score(y_test_fa, y_pred_fa)
rmse_fa = np.sqrt(mean_squared_error(y_test_fa, y_pred_fa))

# Metrics on dollar scale
y_pred_fa_usd = np.exp(y_pred_fa)
y_test_fa_usd = np.exp(y_test_fa)
mape_fa     = np.mean(np.abs((y_test_fa_usd - y_pred_fa_usd) / y_test_fa_usd)) * 100
rmse_fa_usd = np.sqrt(mean_squared_error(y_test_fa_usd, y_pred_fa_usd))

print(f"Ridge on 7 FA factors:")
print(f"  α (CV-selected) : {ridge_fa.alpha_:.4f}")
print(f"  R² (log scale)  : {r2_fa:.4f}")
print(f"  RMSE (log)      : {rmse_fa:.4f}")
print(f"  MAPE            : {mape_fa:.1f}%")
print(f"  RMSE ($)        : ${rmse_fa_usd:,.0f}")


Ridge on 7 FA factors:
  α (CV-selected) : 23.1013
  R² (log scale)  : 0.8139
  RMSE (log)      : 0.1793
  MAPE            : 12.3%
  RMSE ($)        : $78,582


In [49]:
# Build comparison table
comparison = pd.DataFrame({
    'Model':          ['FA-only (7 factors)',  'Ridge (full)',      'Lasso (full)'],
    'Features used':  [7,                       X_rf.shape[1],       n_nonzero_lasso],
    'R² (log)':       [r2_fa,                   r2_ridge,            r2_lasso],
    'RMSE (log)':     [rmse_fa,                 rmse_ridge,          rmse_lasso],
    'MAPE (%)':       [mape_fa,                 mape_ridge,          mape_lasso],
    'RMSE ($)':       [rmse_fa_usd,             rmse_ridge_usd,      rmse_lasso_usd],
})

# Format
comparison_display = comparison.copy()
comparison_display['R² (log)']   = comparison_display['R² (log)'].map(lambda x: f'{x:.4f}')
comparison_display['RMSE (log)'] = comparison_display['RMSE (log)'].map(lambda x: f'{x:.4f}')
comparison_display['MAPE (%)']   = comparison_display['MAPE (%)'].map(lambda x: f'{x:.1f}%')
comparison_display['RMSE ($)']   = comparison_display['RMSE ($)'].map(lambda x: f'${x:,.0f}')

print(comparison_display.to_string(index=False))
print()
print(f"R² gap (Full Ridge vs FA-only): {r2_ridge - r2_fa:.3f}")
print(f"R² gap (Full Lasso vs FA-only): {r2_lasso - r2_fa:.3f}")

              Model  Features used R² (log) RMSE (log) MAPE (%) RMSE ($)
FA-only (7 factors)              7   0.8139     0.1793    12.3%  $78,582
       Ridge (full)            224   0.8841     0.1415     9.2%  $48,242
       Lasso (full)            107   0.8832     0.1421     9.2%  $51,772

R² gap (Full Ridge vs FA-only): 0.070
R² gap (Full Lasso vs FA-only): 0.069


In [50]:
# Visualize actual vs predicted for the FA-only model
fig, ax = plt.subplots(figsize=(7, 6))

ax.scatter(y_test_fa_usd / 1000, y_pred_fa_usd / 1000,
           alpha=0.4, s=15, color='steelblue', rasterized=True)
lo = min(y_test_fa_usd.min(), y_pred_fa_usd.min()) / 1000 * 0.93
hi = max(y_test_fa_usd.max(), y_pred_fa_usd.max()) / 1000 * 1.05
ax.plot([lo, hi], [lo, hi], 'r--', linewidth=1.5, label='Perfect prediction')
ax.set_xlabel('Actual Sale Price ($k)')
ax.set_ylabel('Predicted Sale Price ($k)')
ax.set_title(f'Figure 10 — FA-Only Regression: Actual vs Predicted\n'
             f'R² = {r2_fa:.3f}  |  MAPE = {mape_fa:.1f}%  |  7 latent factors only',
             fontsize=11)
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig('figures/fig10_fa_only_predictions.png', dpi=300, bbox_inches='tight')
plt.close()